# Integrated Variant Prioritization

This is the main notebook for the CFDE lesson. Join the evidence layers while keeping biological context, data coverage, and target development separate. The result is a shortlist for more research, not a clinical pathogenicity score.

In [ ]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("data") if Path("data").exists() else Path("../data")
variants = pd.read_csv(DATA_DIR / "variants.csv")
gtex = pd.read_csv(DATA_DIR / "gtex_expression.csv")
hubmap = pd.read_csv(DATA_DIR / "hubmap_cell_expression.csv")
pharos = pd.read_csv(DATA_DIR / "pharos_target_context.csv")

## Build the evidence matrix

The join uses one named tissue comparison and one cell type. It keeps missing HuBMAP values visible.

In [ ]:
gtex_wide = gtex.pivot(
    index="gene_symbol",
    columns="tissue_id",
    values="median_tpm",
).rename(
    columns={
        "Heart_Atrial_Appendage": "gtex_atrial_tpm",
        "Heart_Left_Ventricle": "gtex_ventricle_tpm",
    }
)

ventricular = (
    hubmap[
        hubmap["cell_type_id"] == "CL:0002131"
    ]
    .loc[
        :,
        [
            "gene_symbol",
            "mean_normalized_expression",
            "percent_detected",
            "availability",
        ],
    ]
    .rename(
        columns={
            "mean_normalized_expression": "hubmap_ventricular_mean",
            "percent_detected": "hubmap_ventricular_percent_detected",
            "availability": "hubmap_availability",
        }
    )
)

In [ ]:
evidence_matrix = (
    variants.loc[:, ["gene_symbol", "hgvs_p", "condition"]]
    .merge(gtex_wide, on="gene_symbol", how="left", validate="one_to_one")
    .merge(ventricular, on="gene_symbol", how="left", validate="one_to_one")
    .merge(
        pharos.loc[:, ["gene_symbol", "tdl", "drug_count"]],
        on="gene_symbol",
        how="left",
        validate="one_to_one",
    )
)
evidence_matrix

## A question-specific shortlist

For a lab follow-up focused on measured heart-muscle-cell context, keep candidates with available HuBMAP values. This rule answers one question. It is not a ranking for every research goal.

In [ ]:
mechanistic_shortlist = (
    evidence_matrix[
        evidence_matrix["hubmap_availability"] == "available"
    ]
    .sort_values("gtex_ventricle_tpm", ascending=False)
    .reset_index(drop=True)
)
mechanistic_shortlist.loc[
    :,
    [
        "gene_symbol",
        "gtex_ventricle_tpm",
        "hubmap_ventricular_percent_detected",
        "tdl",
    ],
]

In [ ]:
coverage_gaps = evidence_matrix[
    evidence_matrix["hubmap_availability"] != "available"
].loc[:, ["gene_symbol", "hubmap_availability"]]
coverage_gaps

## Interpretation template

For one candidate, state the exact variant annotation. Describe the tissue and cell-type results and their limits. Describe Pharos separately. Then name the next experiment or dataset you need. Do not make causal claims that these resources cannot support.